# Tono · consolidar el modelo: barras de error y dos mejoras baratas

> ⚠️ **NO ES UNA HERRAMIENTA DIAGNÓSTICA.** Proyecto educativo y experimental.

El modelo actual detecta el **90,6%** de los melanomas, frente al 84% anterior.
Pero ese número sale de **una sola semilla**, y este proyecto ya se llevó un
susto por eso: una brecha de equidad aparente de 0,237 resultó ser un artefacto
que se disolvió al medir con tres.

Tres cosas, de mayor a menor apalancamiento:

| | Qué | Por qué |
|---|---|---|
| **1** | 3 semillas a 320 px | poner barras de error a la afirmación principal |
| **2** | Promediar volteos (TTA) | una lesión no tiene orientación; promediar reduce varianza sin sesgo |
| **3** | Una tirada a 384 px | 224→320 dio el salto grande; ¿queda más? |

Y un dato de referencia: el **conjunto** de las tres semillas, que marca el techo
alcanzable aunque no se pueda desplegar (triplicaría el peso y el tiempo en el
navegador).

**Métrica que decide: sensibilidad en melanoma**, con umbral derivado en
validación fijando sensibilidad global 0,90.

In [ ]:
import subprocess, sys, os, time, glob, json
T0 = time.time()

for nombre, url in [('tono', 'https://github.com/GGGuardin/tono.git'),
                    ('cxr', 'https://github.com/GGGuardin/chest-xray-pneumonia.git')]:
    subprocess.run(['rm', '-rf', '/tmp/' + nombre], check=False)
    subprocess.run(['git', 'clone', '--depth', '1', '-q', url, '/tmp/' + nombre], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'albumentations'], check=True)

import torch
cap = torch.cuda.get_device_capability(0)
assert 'sm_%d%d' % cap in torch.cuda.get_arch_list(), 'GPU no soportada, relanza con T4'
torch.zeros(8, device='cuda').sum().item()
print('GPU:', torch.cuda.get_device_name(0), '| CUDA OK')

In [ ]:
OUT = '/kaggle/working'
os.chdir('/tmp/tono')

anclas = glob.glob('/kaggle/input/**/fitzpatrick17k*.csv', recursive=True)
FITZ = os.path.dirname(anclas[0])
pad = [p for p in glob.glob('/kaggle/input/**/*.csv', recursive=True)
       if 'fitzpatrick' not in os.path.basename(p).lower()]
PAD = os.path.dirname(pad[0])
while PAD != '/kaggle/input' and not glob.glob(os.path.join(PAD, '**', '*.png'), recursive=True):
    PAD = os.path.dirname(PAD)

!python -m datos.fitzpatrick17k --root {FITZ} --out {OUT}/m_fitz.csv
!python -m datos.pad_ufes --root {PAD} --out {OUT}/m_pad.csv
!python -m scripts.combinar_manifiestos --entradas {OUT}/m_fitz.csv {OUT}/m_pad.csv --out {OUT}/manifiesto.csv

## 1. Tres semillas a 320 px, y una a 384

In [ ]:
SEMILLAS = [42, 1337, 2024]
M = OUT + '/manifiesto.csv'
os.chdir('/tmp/cxr')

for s in SEMILLAS:
    print('===== 320 px, semilla %d =====' % s, flush=True)
    !python -m src.train --config /tmp/tono/configs/combinado_320.yaml --manifest {M} --out-dir {OUT}/runs/s{s} --seed {s}

print('===== 384 px =====', flush=True)
!python -m src.train --config /tmp/tono/configs/combinado_384.yaml --manifest {M} --out-dir {OUT}/runs/px384

## 2. Evaluaciones: normal y con volteos

In [ ]:
MODELOS = {('s%d' % s): OUT + '/runs/s%d/best.pth' % s for s in SEMILLAS}
MODELOS['px384'] = OUT + '/runs/px384/best.pth'

for nombre, ck in MODELOS.items():
    for split in ['val', 'test']:
        !python -m src.evaluate --checkpoint {ck} --manifest {M} --split {split} --out-dir {OUT}/rep/{nombre}_{split} --n-boot 200

In [ ]:
os.chdir('/tmp/tono')
for nombre in ['s42', 'px384']:
    for split in ['val', 'test']:
        !python scripts/evaluar_tta.py --checkpoint {MODELOS[nombre]} --manifest {M} --split {split} --out {OUT}/rep/{nombre}tta_{split}/predictions.csv
os.chdir('/tmp/cxr')

## 3. Comparación

In [ ]:
import pandas as pd, numpy as np
sys.path.insert(0, '/tmp/cxr')
from src.metrics import binary_metrics
from sklearn.metrics import roc_curve

fz = pd.read_csv(OUT + '/m_fitz.csv')[['image_path', 'view']]
fz.columns = ['image_path', 'categoria']

def umbral_sens(y, p, objetivo=0.90):
    _, tpr, u = roc_curve(y, p)
    ok = np.where(tpr >= objetivo)[0]
    return float(u[ok[0]]) if len(ok) else 0.5

def evaluar(val, test):
    u = umbral_sens(val.label.values, val.prob.values, 0.90)
    m = binary_metrics(test.label.values, test.prob.values, u)
    t = test.merge(fz, on='image_path', how='left')
    mel = t[(t.label == 1) & (t.categoria == 'malignant melanoma')]
    return {'umbral': round(u, 4), 'auroc': round(m['auroc'], 4),
            'sens_global': round(m['sensibilidad'], 4),
            'especificidad': round(m['especificidad'], 4),
            'sens_melanoma': round(float((mel.prob >= u).mean()), 4) if len(mel) else None,
            'n_melanoma': int(len(mel))}

def cargar(nombre, split):
    return pd.read_csv(OUT + '/rep/%s_%s/predictions.csv' % (nombre, split))

res = {}
for nombre in list(MODELOS) + ['s42tta', 'px384tta']:
    try:
        res[nombre] = evaluar(cargar(nombre, 'val'), cargar(nombre, 'test'))
    except FileNotFoundError:
        print('falta', nombre)

# Conjunto de las tres semillas: media de sus probabilidades
val_c = cargar('s42', 'val').copy(); test_c = cargar('s42', 'test').copy()
val_c['prob'] = np.mean([cargar('s%d' % s, 'val').prob.values for s in SEMILLAS], axis=0)
test_c['prob'] = np.mean([cargar('s%d' % s, 'test').prob.values for s in SEMILLAS], axis=0)
res['conjunto_3'] = evaluar(val_c, test_c)

print('%-12s %-9s %-11s %-13s %s' % ('', 'AUROC', 'especif.', 'sens.global', 'SENS.MELANOMA'))
for n, r in res.items():
    print('%-12s %-9.4f %-11.4f %-13.4f %s' % (n, r['auroc'], r['especificidad'],
                                               r['sens_global'], r['sens_melanoma']))

In [ ]:
sem = [res['s%d' % s] for s in SEMILLAS]
mel = [r['sens_melanoma'] for r in sem]
auc = [r['auroc'] for r in sem]
variabilidad = {
    'sens_melanoma_media': round(float(np.mean(mel)), 4),
    'sens_melanoma_desv': round(float(np.std(mel, ddof=1)), 4),
    'sens_melanoma_valores': mel,
    'auroc_media': round(float(np.mean(auc)), 4),
    'auroc_desv': round(float(np.std(auc, ddof=1)), 4),
}
print('Sensibilidad en melanoma: %.4f +- %.4f   valores %s'
      % (variabilidad['sens_melanoma_media'], variabilidad['sens_melanoma_desv'], mel))
print('AUROC:                    %.4f +- %.4f' % (variabilidad['auroc_media'], variabilidad['auroc_desv']))
print()
print('Con una sola semilla se reporto 0.906. Si la desviacion es grande,')
print('esa cifra era optimista y hay que dar la media con su margen.')

In [ ]:
mejor = max(res.items(), key=lambda kv: kv[1]['sens_melanoma'] or 0)
desplegables = {k: v for k, v in res.items() if k != 'conjunto_3'}
mejor_desplegable = max(desplegables.items(), key=lambda kv: kv[1]['sens_melanoma'] or 0)

resumen = {
    'objetivo': 'poner barras de error y probar dos mejoras baratas',
    'metrica': 'sensibilidad en melanoma, umbral de validacion a sens. global 0,90',
    'resultados': res,
    'variabilidad_entre_semillas': variabilidad,
    'mejor_absoluto': mejor[0],
    'mejor_desplegable': mejor_desplegable[0],
    'nota_conjunto': 'el conjunto de 3 no es desplegable: triplica peso y tiempo en el navegador',
    'minutos': round((time.time() - T0) / 60, 1),
}
json.dump(resumen, open(OUT + '/resumen.json', 'w'), indent=2, ensure_ascii=False)
print(json.dumps(resumen, indent=2, ensure_ascii=False)[:2600])

import shutil
for f_ in glob.glob(OUT + '/m_*.csv') + glob.glob(OUT + '/manifiesto*.csv'):
    shutil.move(f_, '/tmp/' + os.path.basename(f_))
# Se conservan solo los checkpoints que podrian desplegarse
for f_ in glob.glob(OUT + '/runs/*/best.pth'):
    if not any(k in f_ for k in ['s42', 'px384']):
        os.remove(f_)